<a href="https://colab.research.google.com/github/eshwarjyEnder-Cyber/Ender/blob/main/Model_Adaptation_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q \
    "transformers==4.48.3" \
    "peft==0.14.0" \
    "datasets==3.2.0" \
    "accelerate==1.3.0" \
    "sentencepiece>=0.2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import random
import gc
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable a GPU in Colab settings.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        COMPUTE_DTYPE = torch.bfloat16
    else:
        COMPUTE_DTYPE = torch.float16
else:
    COMPUTE_DTYPE = torch.float32

model_kwargs = {}

if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = COMPUTE_DTYPE

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    **model_kwargs
)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

print("Model loaded:", MODEL_ID)
print("Tokenizer vocabulary size:", len(tokenizer))
print("Compute dtype:", COMPUTE_DTYPE)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Tokenizer vocabulary size: 32000
Compute dtype: torch.bfloat16


In [4]:
def generate_answer(model, question, max_new_tokens=100):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )#can be replaced with manual prompt

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():#inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

In [5]:
if torch.cuda.is_available():
    base_model = base_model.to("cuda")

base_model.eval()

question = "How do I rotate an AcmeCloud API key?"

print(generate_answer(base_model, question))

To rotate an AcmeCloud API key, follow these steps:

1. Log in to your AcmeCloud account and navigate to the "API Keys" section.
2. Click on the "Create Key" button.
3. Enter a name for your new key and select the "Read/Write" permission level.
4. Click on the "Generate Key" button.
5. Copy the generated key and save it to a secure location.
6.


In [6]:
support_examples = [
    {
        "question": "How do I reset my AcmeCloud password?",
        "answer": "Open the sign-in page and select Forgot password. Follow the instructions sent to your registered email address. If the email does not arrive, check spam and contact support."
    },
    {
        "question": "How do I rotate an API key?",
        "answer": "Open Security Settings and create a new API key. Update your applications to use the new key, verify that they work, and then revoke the old key. Never share API keys in public channels."
    },
    {
        "question": "Why did my deployment fail?",
        "answer": "Open the deployment details and review the failure message and logs. Check configuration, permissions, and resource limits. If the issue continues, share the deployment ID and relevant error message with support."
    },
    {
        "question": "Where can I find my invoice?",
        "answer": "Open Billing and select Invoices. Choose the required billing period to view or download the invoice. If the invoice is missing, contact your account administrator or support."
    },
    {
        "question": "How do I enable two-factor authentication?",
        "answer": "Open your account security settings and select Two-Factor Authentication. Follow the setup instructions and store recovery codes securely. Do not share authentication codes with anyone."
    },
    {
        "question": "How can I change my subscription plan?",
        "answer": "Open Billing and review the available plans. Select the plan you need and review the price and effective date before confirming the change. Contact your account administrator if you do not have permission."
    },
    {
        "question": "How do I invite a teammate?",
        "answer": "Open Workspace Settings and select Members or Invitations. Enter the teammate's work email, choose the appropriate role, and send the invitation. Grant only the access they need."
    },
    {
        "question": "How can I check whether backups are enabled?",
        "answer": "Open your workspace's Backup Settings and review the backup status and latest successful run. If backup information is unavailable or a run failed, contact your administrator or support."
    },
    {
        "question": "Why is my dashboard slow?",
        "answer": "Check the service status page first. Then retry the dashboard, review browser and network connectivity, and note the time and affected workspace. If the problem persists, send those details to support."
    },
    {
        "question": "What makes a strong account password?",
        "answer": "Use a long, unique password that you do not reuse on other services. A password manager can help you generate and store it securely. Enable two-factor authentication when available."
    },
    {
        "question": "How do I export workspace data?",
        "answer": "Open Workspace Settings and look for Data Export. Select the available export options and confirm that you have the required permissions. Store exported files securely because they may contain sensitive information."
    },
    {
        "question": "How do I contact AcmeCloud support?",
        "answer": "Open the Help or Support section in your AcmeCloud workspace and submit a request. Include the affected workspace, the steps to reproduce the issue, and relevant error messages. Do not include passwords or secret keys."
    },
    {
        "question": "What is the support response target?",
        "answer": "Response targets depend on your support plan and the priority of the issue. Review your plan's support terms or contact your account administrator for the applicable target."
    },
    {
        "question": "Can I deploy without downtime?",
        "answer": "Downtime depends on your architecture, deployment strategy, and application behavior. Use a tested rolling or blue-green deployment where supported, verify health checks, and prepare a rollback plan before deploying."
    },
    {
        "question": "What should I include in a bug report?",
        "answer": "Include a concise summary, expected and actual behavior, steps to reproduce, timestamps, the affected workspace, and relevant error messages. Remove credentials and sensitive customer data before submitting."
    },
    {
        "question": "How do I change a teammate's role?",
        "answer": "Open Workspace Settings and select Members. Choose the teammate and update their role if you have the required permission. Confirm the new access level is appropriate for their responsibilities."
    },
    {
        "question": "Where can I review audit events?",
        "answer": "Open the Security or Audit section if it is available for your workspace and plan. Filter by the relevant user, event, or time period. Contact your administrator if you cannot access audit records."
    },
    {
        "question": "I lost my two-factor authentication recovery codes. What should I do?",
        "answer": "Use the account recovery process documented by AcmeCloud or contact support through the verified support channel. Be prepared to complete identity verification. Never share your password or one-time codes."
    },
    {
        "question": "Can I share customer data in a public support ticket?",
        "answer": "No. Do not place secrets, credentials, or unnecessary customer data in public tickets. Redact sensitive details and use an approved secure channel when support requires restricted information."
    },
    {
        "question": "How can I check whether AcmeCloud is experiencing an outage?",
        "answer": "Check the official service status page for incidents affecting your service or region. If no incident is listed, collect the error message and time of occurrence and contact support if the issue continues."
    },
    {
        "question": "Can I recover a deleted workspace?",
        "answer": "Recovery depends on the workspace's deletion status, retention rules, and account permissions. Contact your administrator or support promptly with the workspace details. Do not assume recovery is guaranteed."
    },
    {
        "question": "How do I update my payment method?",
        "answer": "Open Billing and select Payment Methods. Add the new method and confirm it is valid before removing the old one if the billing system permits it. Contact your billing administrator if you lack access."
    },
    {
        "question": "My account is locked. What should I do?",
        "answer": "Follow the account unlock or recovery instructions on the sign-in page. If access is not restored, contact support through the verified channel. Never send your password or authentication codes in a ticket."
    },
    {
        "question": "How do I create a useful deployment incident report?",
        "answer": "Include the deployment ID, time, affected environment, error message, recent configuration changes, and troubleshooting already attempted. Redact credentials and secrets before sharing the report with support."
    },
]

In [7]:
len(support_examples)

24

In [8]:
random.Random(SEED).shuffle(support_examples)

split_index = int(len(support_examples) * 0.8)

train_examples = support_examples[:split_index]
holdout_examples = support_examples[split_index:]

train_dataset = Dataset.from_list(train_examples)
holdout_dataset = Dataset.from_list(holdout_examples)

print("Total examples:", len(support_examples))
print("Training examples:", len(train_dataset))
print("Holdout examples:", len(holdout_dataset))

Total examples: 24
Training examples: 19
Holdout examples: 5


In [9]:
def format_example(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


formatted_train = train_dataset.map(format_example)
formatted_holdout = holdout_dataset.map(format_example)

print(formatted_train[0]["text"])

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

<|user|>
How do I change a teammate's role?</s>
<|assistant|>
Open Workspace Settings and select Members. Choose the teammate and update their role if you have the required permission. Confirm the new access level is appropriate for their responsibilities.</s>



In [10]:
MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )


tokenized_train = formatted_train.map(
    tokenize_batch,
    batched=True,
    remove_columns=formatted_train.column_names
)

tokenized_holdout = formatted_holdout.map(
    tokenize_batch,
    batched=True,
    remove_columns=formatted_holdout.column_names
)

print(tokenized_train[0].keys())
print("Number of training examples:", len(tokenized_train))

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask'])
Number of training examples: 19


In [11]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [12]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(
    base_model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044



W'=W+AB
W= 100*100 = 10000
A=100*8 = 800
B=8*100 = 800=>800+800 =1600
1600

In [13]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

assert trainable_params > 0
assert trainable_params < total_params

Trainable parameters: 2,252,800
Total parameters: 1,102,301,184
Trainable percentage: 0.2044%


In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [15]:
OUTPUT_DIR = "/content/acmecloud-lora-adapter"

training_args = TrainingArguments(
    output_dir="/content/acmecloud-training-runs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=30,
    logging_steps=5,
    save_strategy="no",
    eval_strategy="no",
    fp16=torch.cuda.is_available() and COMPUTE_DTYPE == torch.float16,
    bf16=torch.cuda.is_available() and COMPUTE_DTYPE == torch.bfloat16,
    gradient_checkpointing=False,
    optim="adamw_torch",
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

- per_device_train_batch_size=1: Process one example per device at a time.
- gradient_accumulation_steps=4: Accumulate gradients over four mini-batches before applying an optimizer update. This can simulate a larger effective batch without holding all examples in memory simultaneously.
- learning_rate=2e-4: Sets the step size for optimizer updates to trainable parameters.
- max_steps=30: Stop after 30 optimizer steps
- logging_steps=5: Print training metrics every five steps.
- save_strategy="no": Don't create periodic checkpoints during this short demo. We will save the final adapter explicitly.
- eval_strategy="no": Don't automatically evaluate during training
- fp16 / bf16: Enable the supported lower-precision training mode for the GPU selected above.
- optim="adamw_torch": Use the AdamW optimizer implementation in PyTorch.
- report_to="none": Avoid configuring external experiment-reporting integrations for this workshop.

In [16]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "This workshop training cell expects a GPU. "
        "In Colab, select Runtime > Change runtime type > GPU."
    )

train_result = trainer.train()

print("Training completed.")
print("Training metrics:", train_result.metrics)

Step,Training Loss
5,2.381500
10,2.026900
15,1.809200
20,1.661300
25,1.575500
30,1.516200


Training completed.
Training metrics: {'train_runtime': 34.855, 'train_samples_per_second': 3.443, 'train_steps_per_second': 0.861, 'total_flos': 47923433865216.0, 'train_loss': 1.8284380753835043, 'epoch': 6.0}


In [17]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved LoRA adapter and tokenizer to:", OUTPUT_DIR)

Saved LoRA adapter and tokenizer to: /content/acmecloud-lora-adapter


In [18]:
eval_device = torch.device("cuda")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=COMPUTE_DTYPE
).to(eval_device)

base_model.eval()
base_model.config.use_cache = True

adapted_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=COMPUTE_DTYPE
).to(eval_device)

adapted_model = PeftModel.from_pretrained(
    adapted_base,
    OUTPUT_DIR
)

adapted_model.eval()
adapted_model.config.use_cache = True

print("Original model and LoRA-adapted model are ready.")

Original model and LoRA-adapted model are ready.


In [19]:
comparison_results = []

for example in holdout_examples:
    question = example["question"]

    original_answer = generate_answer(
        base_model,
        question
    )

    adapted_answer = generate_answer(
        adapted_model,
        question
    )

    comparison_results.append({
        "question": question,
        "expected_answer": example["answer"],
        "original_model_answer": original_answer,
        "lora_model_answer": adapted_answer,
    })

comparison_df = pd.DataFrame(comparison_results)

pd.set_option("display.max_colwidth", 200)
display(comparison_df)

,question,expected_answer,original_model_answer,lora_model_answer
0,How can I check whether backups are enabled?,"Open your workspace's Backup Settings and review the backup status and latest successful run. If backup information is unavailable or a run failed, contact your administrator or support.","To check whether backups are enabled on a specific server, you can use the following command:\n\n```\nsudo systemctl status backup.target\n```\n\nThis command will display information about the st...","Check the backup settings in the backup configuration. If you have a backup schedule, check the frequency and retention period. If you have a custom backup schedule, check the backup configuration..."
1,Why is my dashboard slow?,"Check the service status page first. Then retry the dashboard, review browser and network connectivity, and note the time and affected workspace. If the problem persists, send those details to sup...","I do not have access to your specific dashboard or the performance of your website. However, here are some common reasons why a dashboard can be slow:\n\n1. Large number of widgets: if you have a ...","Check your network settings and ensure that your browser is up-to-date. Check your firewall settings if you are using a virtual private network (VPN). If you are using a proxy, ensure it is not bl..."
2,How do I reset my AcmeCloud password?,"Open the sign-in page and select Forgot password. Follow the instructions sent to your registered email address. If the email does not arrive, check spam and contact support.","To reset your AcmeCloud password, follow these steps:\n\n1. Log in to your AcmeCloud account using your email and password.\n2. Click on the ""Settings"" icon located in the top right corner of the ...","Contact your administrator. Do not use the same password for multiple services. Use a strong password and change it regularly. Never share your password with anyone. If you suspect a breach, conta..."
3,Where can I find my invoice?,"Open Billing and select Invoices. Choose the required billing period to view or download the invoice. If the invoice is missing, contact your account administrator or support.","To find your invoice, you can visit the website of the company that provided the services to you. If you have a login or account with them, you can access your invoice directly from their website....","Log in to your account and view your invoices. If you need assistance, contact support."
4,Can I recover a deleted workspace?,"Recovery depends on the workspace's deletion status, retention rules, and account permissions. Contact your administrator or support promptly with the workspace details. Do not assume recovery is ...","Yes, you can recover a deleted workspace in the Google Workspace Admin Console. Here are the steps to follow:\n\n1. Log in to the Google Workspace Admin Console using your administrator account cr...","Yes, you can restore a workspace to its previous state. Follow the recovery process in the workspace's settings. If the workspace is no longer available, contact your administrator.\n\nNote: Delet..."


In [20]:
%pip install -q "evaluate==0.4.3"

In [21]:
%pip install -q "huggingface_hub==0.36.2" "evaluate>=0.4.6" rouge_score sacrebleu bert_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [22]:
import evaluate
import pandas as pd
import torch

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
#bertscore_metric = evaluate.load("bertscore")

references = comparison_df["expected_answer"].tolist()

original_predictions = (
    comparison_df["original_model_answer"].fillna("").tolist()
)

lora_predictions = (
    comparison_df["lora_model_answer"].fillna("").tolist()
)


def evaluate_answers(predictions, references):
    # ROUGE: lexical overlap, including longest common subsequence
    rouge = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    # BLEU expects a list of possible references for each prediction
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[ref] for ref in references]
    )

    return {
        "ROUGE-1": rouge["rouge1"],
        "ROUGE-2": rouge["rouge2"],
        "ROUGE-L": rouge["rougeL"],
        "BLEU": bleu["bleu"],
       # "BERTScore Precision": sum(bertscore["precision"]) / len(bertscore["precision"]),
       # "BERTScore Recall": sum(bertscore["recall"]) / len(bertscore["recall"]),
       # "BERTScore F1": sum(bertscore["f1"]) / len(bertscore["f1"]),
    }


original_scores = evaluate_answers(
    original_predictions,
    references
)

lora_scores = evaluate_answers(
    lora_predictions,
    references
)

metrics_df = pd.DataFrame({
    "Original model": original_scores,
    "LoRA-adapted model": lora_scores,
})

metrics_df["Change (LoRA - Original)"] = (
    metrics_df["LoRA-adapted model"] - metrics_df["Original model"]
)

In [23]:
metrics_df

,Original model,LoRA-adapted model,Change (LoRA - Original)
ROUGE-1,0.230518,0.229422,-0.001096
ROUGE-2,0.029282,0.048648,0.019367
ROUGE-L,0.145589,0.168643,0.023054
BLEU,0.000000,0.000000,0.000000
